# Align spot intervals and open-interest observations

## Goal

Attach available, recent open-interest observations to finished five-minute spot intervals. Preserve nulls for missing or stale data; infer no funding rate or trading signal.

This notebook uses synthetic teaching data, not a paper replication or production observations.

## Setup

Use a Python 3.10+ kernel and run all cells in order. Computation uses only the standard library, without keys, networking or extra data files. Open in an existing Jupyter environment.

Embedded inputs match inputs.json in the same download directory. Edit args in the next cell to experiment; preserve explicit times and units.

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"crypto-observation-alignment\",\"identity\":\"synthetic\",\"args\":[[{\"openTime\":\"2025-01-06T00:00:00Z\",\"endExclusive\":\"2025-01-06T00:05:00Z\",\"close\":100},{\"openTime\":\"2025-01-06T00:05:00Z\",\"endExclusive\":\"2025-01-06T00:10:00Z\",\"close\":101},{\"openTime\":\"2025-01-06T00:10:00Z\",\"endExclusive\":\"2025-01-06T00:15:00Z\",\"close\":102}],[{\"observedAt\":\"2025-01-06T00:04:00Z\",\"firstSeenAt\":\"2025-01-06T00:06:00Z\",\"value\":10,\"unit\":\"BTC\"},{\"observedAt\":\"2025-01-06T00:09:00Z\",\"firstSeenAt\":\"2025-01-06T00:09:30Z\",\"value\":12,\"unit\":\"BTC\"}],300,\"BTC\"],\"expected\":[{\"openTime\":\"2025-01-06T00:00:00Z\",\"endExclusive\":\"2025-01-06T00:05:00Z\",\"close\":100,\"oiObservedAt\":null,\"ageSeconds\":null,\"openInterest\":null,\"unit\":\"BTC\",\"status\":\"no_available_observation\"},{\"openTime\":\"2025-01-06T00:05:00Z\",\"endExclusive\":\"2025-01-06T00:10:00Z\",\"close\":101,\"oiObservedAt\":\"2025-01-06T00:09:00Z\",\"ageSeconds\":60,\"openInterest\":12,\"unit\":\"BTC\",\"status\":\"aligned\"},{\"openTime\":\"2025-01-06T00:10:00Z\",\"endExclusive\":\"2025-01-06T00:15:00Z\",\"close\":102,\"oiObservedAt\":\"2025-01-06T00:09:00Z\",\"ageSeconds\":360,\"openInterest\":null,\"unit\":\"BTC\",\"status\":\"stale\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## Steps

### 1. Do not equate similarly named products

Freeze venue, spot pair, perpetual contract, base asset and quote asset before passing one mapped pair to the function. BTC is a teaching unit; all values are synthetic. Open-interest quantity and notional value are different fields, so a USDT amount must not silently replace a BTC quantity.

### 2. Make clock mapping explicit

Normalize explicitly to UTC whole seconds, retaining raw timestamps. Spot intervals are half-open, with the decision time at their end boundary. Open-interest observation time identifies the measurement; first-seen time identifies acquisition. A later-acquired record cannot become historically available just because its measurement is earlier.

### 3. Join backward with an age limit

Choose the newest measurement whose observation and first-seen times are both no later than the interval end. Boundary-time availability is allowed. The 300-second maximum age is a teaching convention, not an exchange guarantee. Older candidates become stale with null values; absent candidates remain explicitly missing.

### 4. Test late arrivals and unit conflicts

The first interval cannot use an observation acquired after its end; the second aligns and the third has stale information. Duplicate observation times, unit mismatches and non-five-minute intervals are rejected. Successful alignment validates only this timing convention, not a causal relation between price and open interest.

### Method and assumptions

- Open interest is a stock, not traded volume; do not sum it across intervals.
- Funding rates, premium indices and open interest are not interchangeable.
- A catalog contract does not establish current availability; check network, grants, coverage and first-observation evidence separately.

In [ ]:
from datetime import datetime, timezone
import math


def align_spot_and_open_interest(bars, observations, max_age_seconds, expected_unit):
    def parse(value):
        try:
            parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if parsed.strftime("%Y-%m-%dT%H:%M:%SZ") != value:
                raise ValueError()
            return int(parsed.timestamp())
        except (ValueError, TypeError):
            raise ValueError("utc_seconds_required")

    def finite(value):
        return not isinstance(value, bool) and isinstance(value, (int, float)) and math.isfinite(value)

    if not finite(max_age_seconds) or int(max_age_seconds) != max_age_seconds or not 0 <= max_age_seconds <= 86400 or not isinstance(expected_unit, str) or not expected_unit:
        raise ValueError("invalid_alignment_contract")
    oi_times, oi = set(), []
    for row in observations:
        time = parse(row.get("observedAt"))
        available = max(time, parse(row.get("firstSeenAt")))
        if time in oi_times:
            raise ValueError("duplicate_observation")
        oi_times.add(time)
        if not finite(row.get("value")) or row["value"] < 0 or row.get("unit") != expected_unit:
            raise ValueError("invalid_open_interest_unit_or_value")
        oi.append((row, time, available))
    oi.sort(key=lambda item: -item[1])
    bar_times, result = set(), []
    for bar in sorted(bars, key=lambda row: parse(row.get("endExclusive"))):
        start, end = parse(bar.get("openTime")), parse(bar.get("endExclusive"))
        if end - start != 300 or start % 300 != 0 or start in bar_times:
            raise ValueError("invalid_or_duplicate_bar")
        bar_times.add(start)
        if not finite(bar.get("close")) or bar["close"] <= 0:
            raise ValueError("invalid_close")
        candidate = next((item for item in oi if item[1] <= end and item[2] <= end), None)
        age = end - candidate[1] if candidate else None
        status = "no_available_observation" if not candidate else "stale" if age > max_age_seconds else "aligned"
        result.append({**bar, "oiObservedAt": candidate[0]["observedAt"] if candidate else None, "ageSeconds": age, "openInterest": candidate[0]["value"] if status == "aligned" else None, "unit": expected_unit, "status": status})
    return result


### Run the sample

The three intervals are no_available_observation, aligned and stale; open interest is null, 12 and null.

In [ ]:
result = align_spot_and_open_interest(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Checks

Compare every row with the browser example's expected output. After editing inputs, a failed assertion may be expected: explain the difference before changing the check.

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("Passed: output matches the synthetic browser example.")

## Next steps

Before real data, confirm grants, fields, schema_major, windows and provenance using authenticated GET /v1/catalog, then map the actual contract. Candidate IDs below do not guarantee availability or historical completeness. API as_of is not a historical filing-version guarantee. Validate again after substituting real inputs; the synthetic pass does not transfer.

- `crypto.spot.binance.btcusdt.5m`
- `crypto.perp.binance.btcusdt.open_interest`

### References

- [Binance: spot candle timestamps](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/market-data-endpoints)
- [Binance: open-interest statistics](https://developers.binance.com/docs/derivatives/usds-margined-futures/market-data/rest-api/Open-Interest-Statistics)

[Back to tutorial](https://tradingdatas.com/recipes/crypto-observation-alignment/)